# AI Layoff Risk Prediction
Predicting job layoff vulnerability based on industry, role, and workforce metrics using Logistic Regression.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

## 1. Data Ingestion & Overview

In [2]:
x=pd.read_csv(r"C:\Users\Adnan Baismail\Downloads\ai layoffs\ai-impact-jobs-layoff-risk-dataset.csv")

In [3]:
x.shape

(20000, 16)

In [4]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Age                         20000 non-null  int64 
 1   Education_Level             20000 non-null  object
 2   Years_of_Experience         20000 non-null  int64 
 3   Industry                    20000 non-null  object
 4   Job_Role                    20000 non-null  object
 5   Company_Size                20000 non-null  object
 6   Job_Level                   20000 non-null  object
 7   Routine_Task_Percentage     20000 non-null  int64 
 8   Creativity_Requirement      20000 non-null  int64 
 9   Human_Interaction_Level     20000 non-null  int64 
 10  AI_Adoption_Level           20000 non-null  object
 11  Number_of_AI_Tools_Used     20000 non-null  int64 
 12  AI_Usage_Hours_Per_Week     20000 non-null  int64 
 13  Tasks_Automated_Percentage  20000 non-null  in

In [5]:
x.head()

,Age,Education_Level,Years_of_Experience,Industry,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours,Layoff_Risk
0,59,Master's,6,Finance,Accountant,Medium,Entry,84,21,94,Medium,5,7,40,26,High
1,44,Master's,14,Manufacturing,Production Supervisor,Small,Entry,30,84,68,Low,2,2,14,9,Low
2,36,Bachelor's,7,Retail,Store Manager,Medium,Senior,12,86,71,Low,0,1,6,3,Low
3,27,Bachelor's,6,Finance,Auditor,Large,Entry,90,18,21,Medium,3,10,67,26,High
4,49,High School,12,Finance,Auditor,Small,Entry,49,52,72,Medium,5,13,26,19,Medium


## 2. Feature Preprocessing & Encoding
- **Target mapping:** Encode `Layoff_Risk` into numerical classes (`Low: 0`, `Medium: 1`, `High: 2`).
- **Ordinal Encoding:** Preserve natural hierarchies in education, company size, experience, and risk categories.
- **One-Hot Encoding:** Convert nominal categories (`Industry`, `Job_Role`) without assuming order.

In [6]:
x['Layoff_Risk']=x['Layoff_Risk'].map({'Low':0,'Medium':1,'High':2})

In [7]:
x['Education_Level'].dtype

dtype('O')

In [8]:
c_g=[]
for i in x.columns:
    if x[i].dtype=='O':

        print(i,x[i].nunique())
        c_g.append(i)

Education_Level 4
Industry 8
Job_Role 24
Company_Size 3
Job_Level 3
AI_Adoption_Level 3


In [9]:
c_g

['Education_Level',
 'Industry',
 'Job_Role',
 'Company_Size',
 'Job_Level',
 'AI_Adoption_Level']

In [10]:
del(c_g[1])
del(c_g[1])

In [11]:
c_g

['Education_Level', 'Company_Size', 'Job_Level', 'AI_Adoption_Level']

In [12]:
for i in c_g:
    print(i,x[i].unique())

Education_Level ["Master's" "Bachelor's" 'High School' 'PhD']
Company_Size ['Medium' 'Small' 'Large']
Job_Level ['Entry' 'Senior' 'Mid']
AI_Adoption_Level ['Medium' 'Low' 'High']


In [13]:
from sklearn.preprocessing import OrdinalEncoder
O=OrdinalEncoder(categories=[
    ["High School" ,"Bachelor's","Master's" , "PhD"],
    ["Small","Medium","Large"],
    ["Entry","Mid","Senior"],
    ["Low","Medium","High"]
])

In [14]:
x[c_g]=O.fit_transform(x[c_g])

In [15]:
for i in ['Industry','Job_Role']:
    A=pd.get_dummies(x[i],drop_first=True).replace({True:1,False:0})
    x=pd.concat([x,A],axis=1)
    x.drop(columns=i,inplace=True)

C:\Users\Adnan Baismail\AppData\Local\Temp\ipykernel_20324\1190440033.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  A=pd.get_dummies(x[i],drop_first=True).replace({True:1,False:0})
C:\Users\Adnan Baismail\AppData\Local\Temp\ipykernel_20324\1190440033.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  A=pd.get_dummies(x[i],drop_first=True).replace({True:1,False:0})


In [16]:
x.columns

Index(['Age', 'Education_Level', 'Years_of_Experience', 'Company_Size',
       'Job_Level', 'Routine_Task_Percentage', 'Creativity_Requirement',
       'Human_Interaction_Level', 'AI_Adoption_Level',
       'Number_of_AI_Tools_Used', 'AI_Usage_Hours_Per_Week',
       'Tasks_Automated_Percentage', 'AI_Training_Hours', 'Layoff_Risk',
       'Finance', 'Healthcare', 'IT', 'Logistics', 'Manufacturing', 'Retail',
       'Telecom', 'Accountant', 'Auditor', 'Data Analyst', 'Dispatcher',
       'Financial Analyst', 'Health Analyst', 'Inventory Analyst',
       'ML Engineer', 'Medical Assistant', 'Network Engineer', 'Nurse',
       'Operations Analyst', 'Operator', 'Production Supervisor',
       'Quality Engineer', 'Research Assistant', 'Sales Associate',
       'Software Engineer', 'Store Manager', 'Supply Chain Analyst',
       'Support Specialist', 'Teacher', 'Warehouse Manager'],
      dtype='object')

In [17]:
x

,Age,Education_Level,Years_of_Experience,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,...,Production Supervisor,Quality Engineer,Research Assistant,Sales Associate,Software Engineer,Store Manager,Supply Chain Analyst,Support Specialist,Teacher,Warehouse Manager
0,59,2.0,6,1.0,0.0,84,21,94,1.0,5,...,0,0,0,0,0,0,0,0,0,0
1,44,2.0,14,0.0,0.0,30,84,68,0.0,2,...,1,0,0,0,0,0,0,0,0,0
2,36,1.0,7,1.0,2.0,12,86,71,0.0,0,...,0,0,0,0,0,1,0,0,0,0
3,27,1.0,6,2.0,0.0,90,18,21,1.0,3,...,0,0,0,0,0,0,0,0,0,0
4,49,0.0,12,0.0,0.0,49,52,72,1.0,5,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,56,1.0,10,0.0,0.0,29,62,44,0.0,0,...,0,0,0,1,0,0,0,0,0,0
19996,43,1.0,9,1.0,0.0,87,20,20,2.0,9,...,1,0,0,0,0,0,0,0,0,0
19997,28,0.0,7,1.0,0.0,91,2,49,1.0,2,...,0,0,0,0,0,0,0,1,0,0
19998,46,1.0,3,1.0,2.0,76,40,89,1.0,3,...,0,1,0,0,0,0,0,0,0,0


## 3. Dataset Splitting & Feature Scaling
We use `stratify=y` to maintain the class distribution balance across train and test sets.

In [20]:
X=x.drop(columns=["Layoff_Risk"])

In [21]:
Y=x['Layoff_Risk']

In [22]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [23]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## 4. Model Training & Evaluation

In [25]:
lr=LogisticRegression()
lr.fit(X_train,Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [26]:
Y_pred = lr.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1296
           1       0.91      0.89      0.90      1329
           2       0.95      0.95      0.95      1375

    accuracy                           0.93      4000
   macro avg       0.93      0.93      0.93      4000
weighted avg       0.93      0.93      0.93      4000

